

![landlab-logo](./medias/landlab_header.png)

# Introduction to Landlab 

# Landlab

Landlab is an open-source Python-language package for numerical modeling of Earth surface dynamics.
It contains:

*  A **gridding engine** that represents the model domain. Regular and irregular grids are supported.
*  A **library of process components**, each of which represents a physical process (e.g., generation
   of rain, erosion by flowing water). These components have a common interface and can be combined
   based on a user's needs.
*  Utilities that support general **numerical methods**, file **input/output**, and **visualization**.

# Introduction to Landlab: Grids and simple 2D models

This tutorial will introduce you to the basics of Landlab grids. By the end, you will have a basic understanding of the following:

- The elements that comprise a landlab grid
- The numbering of grid elements 
- How to instantiate different types and sizes of landlab grids
- How to attach fields to grids and set boundary conditions
- How to perform basic calculations across the grid

The tutorial concludes with an (optional / time-permitting) example of how we can rapidly construct a simple, two-dimensional diffusion model on a Landlab raster grid. 

Time-permitting, we may also learn how to instantiate a component that will replicate the diffusion model for us.

## What types of problems can Landlab solve?

Landlab is great for a variety of earth science problems that have one thing in common: routing a flow across a grid. In today's clinic, we'll see how Landlab handles the gradient calculations that are central to driving many earth (or planetary!) surface processes.

![grids](./medias/flow_examples.png)

## What you need to know about Landlab grids

Landlab model grids are 2D data structures that represent the model domain. A few things to know about grid management:

- Grids are Python <i>objects</i>
- Grids use flat arrays
- Grids are comprised of <i>elements</i> such as nodes and links (see Figure)
- Grids are generated from the user-specified geometry of nodes
- Data fields can be attached to grid elements
- Methods are functions to perform operations on the data fields
- There are regular (raster, radial, and hexgonal) and irregular (Voronoi-Delauney) grid types
- Grids have some built-in numerical functions, such as gradient and divergence

![grids](./medias/Grids1.png)

**Figure** Geometry and topology of grid elements on various Landlab grids ([Hobley et al. 2017](https://esurf.copernicus.org/articles/5/21/2017/))


### 👉 [Interactive sketchbook](https://landlab.github.io/grid-sketchbook/)

### Grid elements

As we see in the above figure, Landlab grids are composed of six different grid elements:
*node*, *links*, *patches*, *corners*, *faces*, and *cells*. The most popular of these
are *nodes*, *links*, and *cells*.

In brief,
* *nodes* are points that have *x* and *y* coordinates.
* *links* are edges that connect two *nodes*.
* *cells* are polygons that surround single *nodes*.

### Explore the Landlab grids

First let's look at the different types of grids Landlab supports. The most common is the `RasterModelGrid`, but Landlab offers other grid types useful for different applications. We'll start by importing a couple of different grid types, and seeing how we can create new grids from those types.

The following code imports several grids as well as a function we will use to have a quick look at what these grids look like.

---

**More complete descriptions of these grid types can be found in [Landlab's documentation](https://landlab.csdms.io/user_guide/reference/grid.html)**

---

In [ ]:
from landlab import RasterModelGrid
from landlab.plot.graph import plot_graph

These are all Python <i>classes</i>, and the instances we create of those classes will be our grid <i>objects</i>. For starters, we'll get some basic information on `RasterModelGrid`. Then we'll create an instance of the class `RasterModelGrid` with 4 rows, 5 columns, and 10.0-unit grid spacing in the *x* (column) direction and 5.0 in the *y* (row) direction.

#### Nodes

*nodes* are simply points that have *x* and *y* coordinates. Different grid types lay out
*nodes* in different ways.

Below we create a `RasterModelGrid` that has four rows and 5 columns of nodes.

In [ ]:
grid = RasterModelGrid((4, 5), xy_spacing=(10, 5))
plot_graph(grid, at="node")

The grid has also created a data structure that stores the *x* and *y* coordinates for each
*node*: `xy_of_node` (you can also use `x_of_node` and `y_of_node`, which simply point to the
respective columns of `xy_of_node`).

You access these data structures as attributes of the grid (regardless of the grid type).

In [ ]:
grid.xy_of_node

In [ ]:
grid.x_of_node, grid.y_of_node

Now it's your turn to build a new grid! Create a raster grid that has 5 rows and 4 columns and use a spacing of 10 and plot it. 

In [ ]:
# create your raster

<details>
    <summary>👉 <b>click to see solution</b></summary>

```python

grid_dim = (5,4) 
dx = 10

grid = RasterModelGrid(grid_dim, dx) 
plot_graph(grid, at="node")

```
</details>

## Attaching data to a grid

You can attach data to any *Landlab* model grid and at any grid element through *Landlab* data
**fields**. Data are set and accessed in the same way for any grid type.

We'll start by attaching some data to a grid's nodes. Below we see two different methods for
adding data to a grid.

In [ ]:
from landlab import RasterModelGrid

grid = RasterModelGrid((4, 5))
grid.add_zeros("foo", at="node")
grid.at_node["bar"] = [1.0] * grid.number_of_nodes

To access the data, we use the `at_node` data structure, which is dictionary whose keys are
field names, and values are *numpy* arrays.

In [ ]:
print(f"All at-node fields: {list(grid.at_node)!r}")

In [ ]:
grid.at_node["foo"]

In [ ]:
# Get the values for the field names, "bar"

<details>
    <summary>👉 <b>click to see solution</b></summary>

```python

grid.at_node["bar"]

```
</details>

Now create a new ``RasterModelGrid`` and attach your data to it and add a field called "topographic__elevation". After this, assign a value at the nodes of this field. Make this grid with 50 rows, 50 colums, using a spacing of 25. Print your result

**Bonus challenge** Can you assign one value for half of the grid and a different value for the other half? If yes, include the result in your report. 

In [ ]:
# create your raster here

<details>
    <summary>👉 <b>click to see solution</b></summary>

```python

grid_dim = (50,50)
dx = 25   

grid = RasterModelGrid(grid_dim, dx) 
z = grid.add_zeros("topographic__elevation", at="node")
grid.at_node['topographic__elevation'] += 2
print(z)




```
</details>

We can create a quick plot of the data field using the ``imshow`` method function of the grid.

In [ ]:
# grid.imshow("topographic__elevation", cmap="terrain")

## Introduction to Landlab Components

A "potluck" is a common American tradition in which each guest brings a dish to share. These contributed dishes usually come in one of a few categories: salads, drinks,  main dishes, desserts. The meal comes together as a collection of components, each of which contributes to fulfilling one of these basic roles. The resulting meal provides guests with a great variety of choices. A guest can compose their own complete and unique meal by combining their own choices for components. And each guest, while contributing their own particular dish, gets to share in the creations of their compatriots. There's a bit of standardization---the dishes are sized to fit on the tables, and their contents are accessible to the usual range of serving utensils---but with that standardization lies a great range of creativity.

Component modeling is a bit like the potluck tradition: one can construct a complete simulation by assembling components to represent the different parts of the system to be modeled. In Landlab, a **component** is a semi-standardized Python *class* that represents a particular process or calculation. Components are not stand-alone programs, but rather are designed to be used within another Python program that creates an integrated model or workflow.

![potluck](./medias/Eat_Alberta_Potluck.jpg)

### Sediment diffusion with a Landlab component

Finally, we'll take a look at how Landlab components can simplify model creation even further.

The Landlab component library is composed of individual, interoperable code packages ("components") that each represent a single Earth surface process. Examples of components include flow routing algorithms, a variety of fluvial processes, and yes, soil processes! 

In this final part of our clinic, we'll make use of Landlab's `LinearDiffuser` component to replicate our diffused fault scarp landscape. This time, we'll let the component do more of the work of field and parameter creation for us.

In [ ]:
from landlab import RasterModelGrid
from landlab.components import LinearDiffuser

Great, we've imported a component which is going to help us build a diffusion model. Before we really start to use this component, though, we want to get some basic information on how it works. This information is stored in component "properties." One such property is `input_var_names`.

In [ ]:
# check out input variable names
LinearDiffuser.input_var_names

Since we know `LinearDiffuser` requires a topographic elevation field in order to be instantiated, we need to create that field. Recall that `topographic__elevation` is tied to grid nodes, so we actually need to create a grid instance, and add the `topographic__elevation` field onto the grid.

Create a new grid instance for the `LinearDiffuser` component to run on. Give you grid 25 rows and colums with a node spacing of 10.0 m.

In [ ]:
# create a new grid here
mg = RasterModelGrid((25, 25), 10)

Great, we now have a grid on which we can implement our diffusion component. Recall, however, our diffusion equation from above:

$$\frac{\partial z}{\partial t} = D \nabla^2 z$$

As before, we need some topographic variation in order to drive diffusion. We'll still need to add our `topographic__elevation` field manually, and we'll need to create a "fault" on the grid by elevating half of the nodes.

In [ ]:
# add a field of zeros called "topographic__elevation" and attach it to the grid nodes
# save the field to an array with a new name
z = mg.add_zeros('topographic__elevation', at='node')

In [ ]:
# now elevate the upper half of the landscape
z[(mg.y_of_node > 120)] = 1

In [ ]:
# now display the landscape
mg.imshow("topographic__elevation")

In [ ]:
# close left and right boundaries
mg.set_closed_boundaries_at_grid_edges(True,False,True,False)

Now, rather than building our own diffusion model from scratch, we'll let Landlab's `LinearDiffuser` component do the work for us. Create an instance of this component named `diffusion` and pass your grid to the component.

In [ ]:
# instantiate the linear diffuser component here
diffusion = LinearDiffuser(mg, linear_diffusivity=0.1)

Great, we now have an instance of our component. Next, use the `run_one_step` method with a timestep `dt` on `diffusion` in a `for` loop in order to evolve our faulted landscape. Run the loop for 25 steps. Use a same timestep, `dt`, of 1000.0.

In [ ]:
# create your for loop here
# hint: you only need two lines of code in this cell to run the model
for _ in range(25):
    diffusion.run_one_step(1000)

Great, now visualize your landscape to see if it looks as you expect.

In [ ]:
# visualize landscape
mg.imshow("topographic__elevation")

Fantastic! Your final output should look very simialr to the diffusion model you created previously, but here you can see that using a Landlab component to simulate the diffusion process has simplified your for loop even further. You can appreciate how valuable this simplicity is, especially if you wanted to couple several components together (for example, flow routing, fluvial erosion, and hillslope diffusion) in order to evolve a more complex landscape.